Script 4 — GRPO: Group Relative Policy Optimization
====================================================
Builds on scripts 1-3. The new question:

  PPO needed four models in memory:
    - Policy π_θ
    - Reference policy π_ref
    - Critic V_φ         ← expensive: another full LM
    - Reward model R_φ

  GRPO's insight: we don't need the critic at all.
  Instead of a learned V(s_t) baseline, use the mean reward
  of a GROUP of responses to the SAME prompt as the baseline.

  For each prompt x, sample G responses:
    y_1, y_2, ..., y_G  ~  π_θ(· | x)

  Compute rewards:
    R_1, R_2, ..., R_G

  The advantage of response i:
    A_i = (R_i - mean(R_1..R_G)) / std(R_1..R_G)

  This is PROMPT-SPECIFIC: the baseline adapts to each prompt.
  No critic. No GAE. No per-token values. No backwards loop.

What changes vs PPO
-------------------
  PPO:  one response per prompt → needs V(s_t) to estimate baseline
  GRPO: G responses per prompt  → uses their statistics as baseline

  PPO advantage:  per-token A_t = GAE(δ_0, ..., δ_T)
  GRPO advantage: per-response A_i = (R_i - mean) / std

  Both still use:
    - Log-prob ratio r_t = π_new / π_old   (clipping)
    - KL penalty against reference model

New in this script
------------------
  1. Sample G=8 responses for the same prompt
  2. Compute rewards for each
  3. Normalize: A_i = (R_i - mean) / std
  4. Show what the group baseline looks like vs flat batch baseline
  5. Compute GRPO loss with clipping (same ε=0.2)
  6. Add KL penalty against reference model (new vs PPO scripts)
  7. One gradient step
  8. Verify

Scenario
--------
  Same tiny LM, vocab=8, prompt="a b", reward=+1 per "c" token.
  G = 8 responses sampled per prompt.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

torch.manual_seed(42)

SEP   = "─" * 66
VOCAB = {0:"PAD", 1:"a", 2:"b", 3:"c", 4:"d", 5:"e", 6:"f", 7:"EOS"}
V_    = len(VOCAB)
D     = 16
MAX_LEN = 10
EPS   = 0.2    # clip range  (same as PPO)
BETA  = 0.04   # KL penalty coefficient
LR    = 3e-3
G     = 8      # group size  ← the key new hyperparameter

def tstr(ids): return " ".join(VOCAB[i] for i in ids)

In [2]:
# ──────────────────────────────────────────────────────────────
# Model  (actor only — NO critic head)
# ──────────────────────────────────────────────────────────────
class PolicyLM(nn.Module):
    """
    Identical to script 1 — just the actor.
    No critic head. This is the point of GRPO.
    """
    def __init__(self):
        super().__init__()
        self.embed   = nn.Embedding(V_, D)
        self.pos_enc = nn.Embedding(MAX_LEN, D)
        layer        = nn.TransformerEncoderLayer(
                           d_model=D, nhead=2, dim_feedforward=32,
                           batch_first=True, dropout=0.0)
        self.tf      = nn.TransformerEncoder(layer, num_layers=1)
        self.actor   = nn.Linear(D, V_)

    def forward(self, token_ids):
        T   = token_ids.shape[1]
        pos = torch.arange(T).unsqueeze(0)
        x   = self.embed(token_ids) + self.pos_enc(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(T)
        h   = self.tf(x, mask=mask, is_causal=True)
        return self.actor(h)   # (1, T, V)

    def get_logprobs(self, prompt, actions):
        """
        Recompute log-probs for a fixed sequence of actions.
        Used during the optimization step.
        Returns log_probs: (len(actions),)
        """
        ids = torch.tensor([prompt])
        lps = []
        for action in actions:
            logits = self(ids)
            lp = F.log_softmax(logits[0, -1, :], dim=-1)[action]
            lps.append(lp)
            ids = torch.cat([ids, torch.tensor([[action]])], dim=1)
        return torch.stack(lps)

model = PolicyLM()
n_params = sum(p.numel() for p in model.parameters())

print(SEP)
print("MODEL — Policy only  (no critic)")
print(SEP)
print(f"  Parameters : {n_params}")
print(f"  Backbone   : 1-layer Transformer  d={D}  heads=2")
print(f"  Actor head : Linear({D}, {V_})  → next-token logits")
print(f"  Critic head: NONE  ← this is GRPO")
print()

──────────────────────────────────────────────────────────────────
MODEL — Policy only  (no critic)
──────────────────────────────────────────────────────────────────
  Parameters : 2648
  Backbone   : 1-layer Transformer  d=16  heads=2
  Actor head : Linear(16, 8)  → next-token logits
  Critic head: NONE  ← this is GRPO



In [3]:
# ──────────────────────────────────────────────────────────────
# PHASE 1 — Sample G responses for the same prompt
# ──────────────────────────────────────────────────────────────
print(SEP)
print(f"PHASE 1 — SAMPLE G={G} RESPONSES FOR THE SAME PROMPT")
print(SEP)
print("""
  This is the key difference from PPO.
  PPO: 1 response per prompt → needs critic to estimate baseline
  GRPO: G responses per prompt → uses group statistics as baseline

  All G responses share the same prompt.
  The group gives us a distribution of rewards for THIS prompt,
  which is a much better baseline than the global batch mean.
""")

PROMPT = [1, 2]   # "a b"
TARGET = 3        # "c"
MAX_GEN = 5

model.eval()

# Freeze reference model (the π_ref for KL penalty)
# In real RLHF this is the SFT model. Here it's π_θ at the start of training.
pi_ref = copy.deepcopy(model)

# Freeze old policy for ratio computation (same as PPO script 3)
pi_old = copy.deepcopy(model)

# Storage for all G rollouts
all_tokens    = []   # list of G lists of token ids
all_log_probs = []   # list of G tensors of log-probs
all_rewards   = []   # list of G scalars

print(f"  {'i':>3}  {'response':>20}  {'R_i':>6}  log_probs")
print(f"  {'─'*3}  {'─'*20}  {'─'*6}  {'─'*30}")

with torch.no_grad():
    for i in range(G):
        ids      = torch.tensor([PROMPT])
        tokens_i = []
        lps_i    = []

        for step in range(MAX_GEN):
            logits   = pi_old(ids)
            lp_all   = F.log_softmax(logits[0, -1, :], dim=-1)
            probs    = lp_all.exp()
            action   = torch.multinomial(probs, 1).item()
            lp       = lp_all[action]

            tokens_i.append(action)
            lps_i.append(lp)
            ids = torch.cat([ids, torch.tensor([[action]])], dim=1)
            if action == 7: break   # EOS

        R_i = sum(1.0 for t in tokens_i if t == TARGET)
        all_tokens.append(tokens_i)
        all_log_probs.append(torch.stack(lps_i))
        all_rewards.append(R_i)

        lp_str = "  ".join(f"{lp.item():.3f}" for lp in lps_i)
        print(f"  {i:>3}  {tstr(tokens_i):>20}  {R_i:>6.1f}  {lp_str}")

print()

──────────────────────────────────────────────────────────────────
PHASE 1 — SAMPLE G=8 RESPONSES FOR THE SAME PROMPT
──────────────────────────────────────────────────────────────────

  This is the key difference from PPO.
  PPO: 1 response per prompt → needs critic to estimate baseline
  GRPO: G responses per prompt → uses group statistics as baseline
 
  All G responses share the same prompt.
  The group gives us a distribution of rewards for THIS prompt,
  which is a much better baseline than the global batch mean.

    i              response     R_i  log_probs
  ───  ────────────────────  ──────  ──────────────────────────────
    0                 f EOS     0.0  -1.398  -2.026
    1             c d d EOS     1.0  -2.863  -2.516  -1.872  -2.452
    2           PAD d e d c     1.0  -1.697  -1.941  -1.787  -2.136  -2.783
    3             c b a EOS     1.0  -2.863  -1.891  -2.671  -2.114
    4             f f c f b     1.0  -1.398  -2.326  -2.715  -1.471  -2.020
    5             

In [4]:
# ──────────────────────────────────────────────────────────────
# PHASE 2 — Group advantage: normalize rewards within the group
# ──────────────────────────────────────────────────────────────
print(SEP)
print("PHASE 2 — GROUP ADVANTAGE  A_i = (R_i - mean) / std")
print(SEP)
print("""
  Instead of V(s_t) as baseline, we use the group's own statistics.

  mean(R) = average reward across all G responses to this prompt
  std(R)  = spread of rewards across the group

  A_i = (R_i - mean) / std

  This is called STANDARDIZATION. It does two things:
    1. Centers around 0 (same as subtracting baseline b)
    2. Scales by std (gradient steps are comparable across prompts)

  A_i > 0: response i was better than the group average
  A_i < 0: response i was worse  than the group average
""")

rewards_t = torch.tensor(all_rewards)
mean_R    = rewards_t.mean()
std_R     = rewards_t.std() + 1e-8   # epsilon for numerical stability

advantages = (rewards_t - mean_R) / std_R

print(f"  Rewards across group : {[f'{r:.1f}' for r in all_rewards]}")
print(f"  mean(R)              = {mean_R.item():.4f}")
print(f"  std(R)               = {std_R.item():.4f}")
print()
print(f"  {'i':>3}  {'response':>20}  {'R_i':>6}  {'A_i':>8}  interpretation")
print(f"  {'─'*3}  {'─'*20}  {'─'*6}  {'─'*8}  {'─'*25}")
for i in range(G):
    interp = "better than group" if advantages[i] > 0 else \
             ("worse than group" if advantages[i] < 0 else "exactly average")
    print(f"  {i:>3}  {tstr(all_tokens[i]):>20}  "
          f"{all_rewards[i]:>6.1f}  {advantages[i].item():>+8.4f}  {interp}")
print()

# Compare with flat batch baseline (script 1 approach)
global_mean = 0.5   # pretend batch mean from other prompts
print(f"  For comparison — flat batch baseline (script 1):")
print(f"    b = {global_mean}  (same for ALL prompts, ALL responses)")
for i in range(G):
    print(f"    i={i}  A = {all_rewards[i]:.1f} - {global_mean} = "
          f"{all_rewards[i] - global_mean:+.1f}")
print()
print("  Group baseline adapts per-prompt. Flat baseline does not.")
print("  If this prompt is easy (high rewards), group baseline reflects that.")
print("  Flat baseline would wrongly signal 'above average' for all responses.")
print()

──────────────────────────────────────────────────────────────────
PHASE 2 — GROUP ADVANTAGE  A_i = (R_i - mean) / std
──────────────────────────────────────────────────────────────────

  Instead of V(s_t) as baseline, we use the group's own statistics.
 
  mean(R) = average reward across all G responses to this prompt
  std(R)  = spread of rewards across the group
 
  A_i = (R_i - mean) / std
 
  This is called STANDARDIZATION. It does two things:
    1. Centers around 0 (same as subtracting baseline b)
    2. Scales by std (gradient steps are comparable across prompts)
 
  A_i > 0: response i was better than the group average
  A_i < 0: response i was worse  than the group average

  Rewards across group : ['0.0', '1.0', '1.0', '1.0', '1.0', '0.0', '0.0', '0.0']
  mean(R)              = 0.5000
  std(R)               = 0.5345

    i              response     R_i       A_i  interpretation
  ───  ────────────────────  ──────  ────────  ─────────────────────────
    0                 f 

In [5]:
# ──────────────────────────────────────────────────────────────
# PHASE 3 — GRPO loss with clipping
# ──────────────────────────────────────────────────────────────
print(SEP)
print("PHASE 3 — GRPO LOSS WITH CLIPPING")
print(SEP)
print(f"""
  Same clipped surrogate as PPO (script 3):

    L_clip_i = -mean_t[ min(r_t · A_i,  clip(r_t, 1-ε, 1+ε) · A_i) ]

  where r_t = π_new(a_t) / π_old(a_t)  per token

  Key difference from PPO:
    PPO : A_t is different for each token  (from GAE)
    GRPO: A_i is the same scalar for every token in response i
          (no per-token credit assignment — the whole response gets one score)

  The loss averages over all G responses AND all tokens within each response.
""")

model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
optimizer.zero_grad()

print(f"  {'i':>3}  {'response':>20}  {'A_i':>8}  "
      f"{'mean_r_t':>9}  {'clipped?':>9}  {'L_clip_i':>10}  {'L_kl_i':>8}")
print(f"  {'─'*3}  {'─'*20}  {'─'*8}  "
      f"{'─'*9}  {'─'*9}  {'─'*10}  {'─'*8}")

total_loss = torch.tensor(0.0)
loss_details = []

for i in range(G):
    tokens_i = all_tokens[i]
    A_i      = advantages[i]
    lp_old_i = all_log_probs[i].detach()   # fixed

    # Recompute log-probs under current π_θ
    lp_new_i = model.get_logprobs(PROMPT, tokens_i)

    # Recompute log-probs under π_ref (for KL penalty)
    with torch.no_grad():
        lp_ref_i = pi_ref.get_logprobs(PROMPT, tokens_i)

    # Probability ratio r_t = π_new / π_old  per token
    ratio_i   = torch.exp(lp_new_i - lp_old_i)            # (T_i,)
    clipped_i = torch.clamp(ratio_i, 1 - EPS, 1 + EPS)

    # Clipped surrogate  (A_i is scalar — broadcast to all tokens)
    unclip_i  = ratio_i  * A_i
    clip_i    = clipped_i * A_i
    was_clipped = (ratio_i.detach() != clipped_i.detach()).any().item()

    actor_loss_i = -torch.min(unclip_i, clip_i).mean()

    # KL penalty: β · KL(π_θ || π_ref) per token
    # KL(p||q) ≈ log(p/q) · p  but simplified as: log π_new - log π_ref
    kl_i = (lp_new_i - lp_ref_i).mean()   # per-token KL approximation
    kl_loss_i = BETA * kl_i

    loss_i = actor_loss_i + kl_loss_i
    total_loss = total_loss + loss_i

    loss_details.append({
        "actor": actor_loss_i.item(),
        "kl":    kl_loss_i.item(),
        "total": loss_i.item(),
    })

    print(f"  {i:>3}  {tstr(tokens_i):>20}  {A_i.item():>+8.4f}  "
          f"{ratio_i.mean().item():>9.4f}  "
          f"{'yes' if was_clipped else 'no':>9}  "
          f"{actor_loss_i.item():>+10.4f}  "
          f"{kl_loss_i.item():>+8.4f}")

total_loss = total_loss / G   # average over group
print()
print(f"  total_loss = sum(L_i) / G = {total_loss.item():+.4f}")
print()

──────────────────────────────────────────────────────────────────
PHASE 3 — GRPO LOSS WITH CLIPPING
──────────────────────────────────────────────────────────────────

  Same clipped surrogate as PPO (script 3):
 
    L_clip_i = -mean_t[ min(r_t · A_i,  clip(r_t, 1-ε, 1+ε) · A_i) ]
 
  where r_t = π_new(a_t) / π_old(a_t)  per token
 
  Key difference from PPO:
    PPO : A_t is different for each token  (from GAE)
    GRPO: A_i is the same scalar for every token in response i
          (no per-token credit assignment — the whole response gets one score)
 
  The loss averages over all G responses AND all tokens within each response.

    i              response       A_i   mean_r_t   clipped?    L_clip_i    L_kl_i
  ───  ────────────────────  ────────  ─────────  ─────────  ──────────  ────────
    0                 f EOS   -0.9354     1.0000         no     +0.9354   -0.0000
    1             c d d EOS   +0.9354     1.0000         no     -0.9354   +0.0000
    2           PAD d e d c   +

In [6]:
# ──────────────────────────────────────────────────────────────
# PHASE 4 — KL penalty explained
# ──────────────────────────────────────────────────────────────
print(SEP)
print("PHASE 4 — KL PENALTY  β · KL(π_θ || π_ref)")
print(SEP)
print(f"""
  In PPO (script 3) we had clipping to prevent large updates.
  In GRPO/RLHF we also add a KL penalty against the reference model π_ref.

  Why?
    Clipping prevents a single UPDATE STEP from being too large.
    KL penalty prevents the policy from DRIFTING too far from π_ref
    over the course of many update steps.

  π_ref is the SFT model — a policy that generates coherent text.
  Without the KL penalty, the policy might learn to maximize reward
  while generating gibberish that happens to score well.

  The penalized reward for response i:
    R̃_i = R_i - β · KL(π_θ(· | x) || π_ref(· | x))

  With β = {BETA}:
    High KL → large penalty → response pushed back toward π_ref
    Low KL  → small penalty → response was already close to reference

  KL per response:
""")
for i in range(G):
    print(f"    i={i}  R_i={all_rewards[i]:.1f}  "
          f"KL_loss={loss_details[i]['kl']:+.4f}  "
          f"actor_loss={loss_details[i]['actor']:+.4f}  "
          f"total={loss_details[i]['total']:+.4f}")
print()

──────────────────────────────────────────────────────────────────
PHASE 4 — KL PENALTY  β · KL(π_θ || π_ref)
──────────────────────────────────────────────────────────────────

  In PPO (script 3) we had clipping to prevent large updates.
  In GRPO/RLHF we also add a KL penalty against the reference model π_ref.
 
  Why?
    Clipping prevents a single UPDATE STEP from being too large.
    KL penalty prevents the policy from DRIFTING too far from π_ref
    over the course of many update steps.
 
  π_ref is the SFT model — a policy that generates coherent text.
  Without the KL penalty, the policy might learn to maximize reward
  while generating gibberish that happens to score well.
 
  The penalized reward for response i:
    R̃_i = R_i - β · KL(π_θ(· | x) || π_ref(· | x))
 
  With β = 0.04:
    High KL → large penalty → response pushed back toward π_ref
    Low KL  → small penalty → response was already close to reference
 
  KL per response:

    i=0  R_i=0.0  KL_loss=-0.0000  actor

In [7]:
# ──────────────────────────────────────────────────────────────
# PHASE 5 — Backward + step
# ──────────────────────────────────────────────────────────────
print(SEP)
print("PHASE 5 — BACKWARD + OPTIMIZER STEP")
print(SEP)

total_loss.backward()

print("  Gradient norms per layer:")
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"    {name:40s}  {param.grad.norm().item():.6f}")
print()
optimizer.step()
print("  optimizer.step() done.")
print()


──────────────────────────────────────────────────────────────────
PHASE 5 — BACKWARD + OPTIMIZER STEP
──────────────────────────────────────────────────────────────────
  Gradient norms per layer:
    embed.weight                              0.056393
    pos_enc.weight                            0.061917
    tf.layers.0.self_attn.in_proj_weight      0.195405
    tf.layers.0.self_attn.in_proj_bias        0.031703
    tf.layers.0.self_attn.out_proj.weight     0.269060
    tf.layers.0.self_attn.out_proj.bias       0.076045
    tf.layers.0.linear1.weight                0.147792
    tf.layers.0.linear1.bias                  0.041564
    tf.layers.0.linear2.weight                0.218066
    tf.layers.0.linear2.bias                  0.114383
    tf.layers.0.norm1.weight                  0.099013
    tf.layers.0.norm1.bias                    0.125424
    tf.layers.0.norm2.weight                  0.113189
    tf.layers.0.norm2.bias                    0.127362
    actor.weight                

In [8]:
# ──────────────────────────────────────────────────────────────
# PHASE 6 — Verification
# ──────────────────────────────────────────────────────────────
print(SEP)
print("PHASE 6 — VERIFICATION")
print(SEP)
print("  Responses with A_i > 0 should have higher log-probs after update.")
print("  Responses with A_i < 0 should have lower log-probs after update.")
print()

model.eval()
print(f"  {'i':>3}  {'response':>20}  {'A_i':>8}  "
      f"{'lp_before':>10}  {'lp_after':>10}  {'Δ':>8}")
print(f"  {'─'*3}  {'─'*20}  {'─'*8}  "
      f"{'─'*10}  {'─'*10}  {'─'*8}")

with torch.no_grad():
    for i in range(G):
        lp_before = all_log_probs[i].sum().item()
        lp_after  = model.get_logprobs(PROMPT, all_tokens[i]).sum().item()
        delta     = lp_after - lp_before
        expected  = "↑" if advantages[i] > 0 else "↓"
        actual    = "↑" if delta > 0 else "↓"
        match     = "✓" if expected == actual else "✗"
        print(f"  {i:>3}  {tstr(all_tokens[i]):>20}  "
              f"{advantages[i].item():>+8.4f}  "
              f"{lp_before:>10.4f}  {lp_after:>10.4f}  "
              f"{delta:>+8.4f} {actual} {match}")
print()

──────────────────────────────────────────────────────────────────
PHASE 6 — VERIFICATION
──────────────────────────────────────────────────────────────────
  Responses with A_i > 0 should have higher log-probs after update.
  Responses with A_i < 0 should have lower log-probs after update.

    i              response       A_i   lp_before    lp_after         Δ
  ───  ────────────────────  ────────  ──────────  ──────────  ────────
    0                 f EOS   -0.9354     -3.4243     -3.7281   -0.3039 ↓ ✓
    1             c d d EOS   +0.9354     -9.7040     -9.4308   +0.2733 ↑ ✓
    2           PAD d e d c   +0.9354    -10.3440    -10.0961   +0.2478 ↑ ✓
    3             c b a EOS   +0.9354     -9.5396     -9.0019   +0.5377 ↑ ✓
    4             f f c f b   +0.9354     -9.9297     -9.9582   -0.0286 ↓ ✗
    5             f a f d f   -0.9354     -7.6354     -7.9373   -0.3019 ↓ ✓
    6           PAD f b EOS   -0.9354     -7.1692     -7.3265   -0.1572 ↓ ✓
    7       PAD e PAD PAD f   -

In [9]:
# ──────────────────────────────────────────────────────────────
# PHASE 7 — PPO vs GRPO side by side
# ──────────────────────────────────────────────────────────────
print(SEP)
print("PPO vs GRPO — SIDE BY SIDE")
print(SEP)
print(f"""
  ┌─────────────────────────┬──────────────────────────────┬──────────────────────────────┐
  │                         │ PPO (scripts 1-3)            │ GRPO (this script)           │
  ├─────────────────────────┼──────────────────────────────┼──────────────────────────────┤
  │ Responses per prompt    │ 1                            │ G = {G}                           │
  │ Baseline                │ V_φ(s_t)  learned critic     │ mean(R_1..R_G)  group stat   │
  │ Advantage               │ A_t per token  (GAE)         │ A_i per response (normalize) │
  │ Credit assignment       │ per token via GAE            │ whole response gets one score│
  │ Extra models needed     │ critic V_φ  (full LM size)   │ none                         │
  │ Memory                  │ 4 models                     │ 2 models (policy + ref)      │
  │ Clipping                │ yes  ε = {EPS}                     │ yes  ε = {EPS}                     │
  │ KL penalty              │ optional                     │ standard  β = {BETA}               │
  │ GAE / TD errors         │ yes                          │ no                           │
  │ G_t / V(s_t) needed     │ yes                          │ no                           │
  └─────────────────────────┴──────────────────────────────┴──────────────────────────────┘

  GRPO loss (full):
    L_GRPO = (1/G) Σ_i (1/T_i) Σ_t [
        -min( r_t · A_i,  clip(r_t, 1-ε, 1+ε) · A_i )   ← clipped actor
        + β · KL( π_θ(·|s_t) || π_ref(·|s_t) )           ← KL penalty
    ]

  where:
    r_t  = π_new(a_t|s_t) / π_old(a_t|s_t)               ← ratio (same as PPO)
    A_i  = (R_i - mean(R_1..R_G)) / std(R_1..R_G)         ← group advantage
    β    = {BETA}                                                ← KL coefficient
""")


──────────────────────────────────────────────────────────────────
PPO vs GRPO — SIDE BY SIDE
──────────────────────────────────────────────────────────────────

  ┌─────────────────────────┬──────────────────────────────┬──────────────────────────────┐
  │                         │ PPO (scripts 1-3)            │ GRPO (this script)           │
  ├─────────────────────────┼──────────────────────────────┼──────────────────────────────┤
  │ Responses per prompt    │ 1                            │ G = 8                           │
  │ Baseline                │ V_φ(s_t)  learned critic     │ mean(R_1..R_G)  group stat   │
  │ Advantage               │ A_t per token  (GAE)         │ A_i per response (normalize) │
  │ Credit assignment       │ per token via GAE            │ whole response gets one score│
  │ Extra models needed     │ critic V_φ  (full LM size)   │ none                         │
  │ Memory                  │ 4 models                     │ 2 models (policy + ref)      │
  │ Cli